# Count Rank Simulation

จำลองการทำงานของฟังก์ชัน `count_rank` จาก C++ เป็น Python  
นับจำนวนแถวที่มีค่าไม่เป็นศูนย์ (ภายใน `col_limit` คอลัมน์แรก) โดย debug ทุก loop ย่อย

In [1]:
# ===== ค่าคงที่ =====
# tolerance สำหรับตัดสินว่า "เป็นศูนย์" หรือไม่
# ค่าที่ abs <= PIVOT_TOL จะถือว่าเป็นศูนย์
PIVOT_TOL = 1e-9

In [2]:
def count_rank(mat, col_limit, pivot_tol=PIVOT_TOL, verbose=True):
    """
    นับจำนวนแถวที่มีค่า nonzero ภายใน col_limit คอลัมน์แรก

    Parameters
    ----------
    mat : list[list[float]]
        เมทริกซ์ 2 มิติ (list of rows)
    col_limit : int
        จำนวนคอลัมน์สูงสุดที่จะตรวจ
    pivot_tol : float
        ค่า tolerance — ถ้า abs(val) <= pivot_tol ถือว่าเป็น 0
    verbose : bool
        ถ้า True จะ print debug ทุก iteration

    Returns
    -------
    int
        จำนวนแถวที่มีค่า nonzero (rank estimate)
    """
    rank = 0

    for i, row in enumerate(mat):
        nonzero = False
        # จำนวนคอลัมน์ที่จะ scan = min(col_limit, ความยาวแถว)
        scan_len = min(col_limit, len(row))

        if verbose:
            print(f"--- row {i}: {row[:scan_len]} (scan {scan_len} cols) ---")

        # === inner loop: วนทีละคอลัมน์ ===
        for j in range(scan_len):
            val = row[j]
            abs_val = abs(val)
            is_above = abs_val > pivot_tol  # True = ไม่เป็นศูนย์

            if verbose:
                # แสดงค่าแต่ละช่องที่ตรวจ
                mark = "✓ nonzero" if is_above else "✗ zero"
                print(f"    [{i},{j}] val={val:>12.6e}  |val|={abs_val:.6e}  {mark}")

            if is_above:
                nonzero = True
                if verbose:
                    print(f"    >> break inner loop (พบค่า nonzero แล้ว)")
                break  # เจอค่า nonzero ตัวเดียวก็พอ

        # === สรุปผลแถวนี้ ===
        if nonzero:
            rank += 1

        if verbose:
            status = "NONZERO ✓" if nonzero else "ALL ZERO ✗"
            print(f"    => row {i}: {status}  |  rank สะสม = {rank}\n")

    if verbose:
        print(f"===== Final rank = {rank} =====")

    return rank

## Test Case 1: เมทริกซ์ 3×3 ปกติ (full rank)
ทุกแถวมีค่า nonzero → คาดว่า rank = 3

In [3]:
mat1 = [
    [1.0, 2.0, 3.0],
    [0.0, 4.0, 5.0],
    [0.0, 0.0, 6.0],
]

# col_limit = 3 → scan ทั้ง 3 คอลัมน์
result = count_rank(mat1, col_limit=3)
assert result == 3, f"Expected 3, got {result}"

--- row 0: [1.0, 2.0, 3.0] (scan 3 cols) ---
    [0,0] val=1.000000e+00  |val|=1.000000e+00  ✓ nonzero
    >> break inner loop (พบค่า nonzero แล้ว)
    => row 0: NONZERO ✓  |  rank สะสม = 1

--- row 1: [0.0, 4.0, 5.0] (scan 3 cols) ---
    [1,0] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    [1,1] val=4.000000e+00  |val|=4.000000e+00  ✓ nonzero
    >> break inner loop (พบค่า nonzero แล้ว)
    => row 1: NONZERO ✓  |  rank สะสม = 2

--- row 2: [0.0, 0.0, 6.0] (scan 3 cols) ---
    [2,0] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    [2,1] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    [2,2] val=6.000000e+00  |val|=6.000000e+00  ✓ nonzero
    >> break inner loop (พบค่า nonzero แล้ว)
    => row 2: NONZERO ✓  |  rank สะสม = 3

===== Final rank = 3 =====


## Test Case 2: มีแถวศูนย์ทั้งหมด
แถวที่ 2 เป็น 0 หมด → คาดว่า rank = 2

In [4]:
mat2 = [
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],   # แถวศูนย์
    [0.0, 0.0, 7.0],
]

result = count_rank(mat2, col_limit=3)
assert result == 2, f"Expected 2, got {result}"

--- row 0: [1.0, 0.0, 0.0] (scan 3 cols) ---
    [0,0] val=1.000000e+00  |val|=1.000000e+00  ✓ nonzero
    >> break inner loop (พบค่า nonzero แล้ว)
    => row 0: NONZERO ✓  |  rank สะสม = 1

--- row 1: [0.0, 0.0, 0.0] (scan 3 cols) ---
    [1,0] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    [1,1] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    [1,2] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    => row 1: ALL ZERO ✗  |  rank สะสม = 1

--- row 2: [0.0, 0.0, 7.0] (scan 3 cols) ---
    [2,0] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    [2,1] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    [2,2] val=7.000000e+00  |val|=7.000000e+00  ✓ nonzero
    >> break inner loop (พบค่า nonzero แล้ว)
    => row 2: NONZERO ✓  |  rank สะสม = 2

===== Final rank = 2 =====


## Test Case 3: col_limit จำกัดการ scan
ถ้า col_limit=1 จะมองแค่คอลัมน์แรก → แถว 1 กับ 2 ที่คอลัมน์ 0 เป็น 0 จะนับไม่ได้

In [5]:
mat3 = [
    [5.0, 0.0, 0.0],   # col 0 = 5 → nonzero
    [0.0, 3.0, 0.0],   # col 0 = 0 → zero (ไม่ได้ดู col 1)
    [0.0, 0.0, 9.0],   # col 0 = 0 → zero (ไม่ได้ดู col 2)
]

# col_limit = 1 → scan แค่คอลัมน์ 0
result = count_rank(mat3, col_limit=1)
assert result == 1, f"Expected 1, got {result}"

--- row 0: [5.0] (scan 1 cols) ---
    [0,0] val=5.000000e+00  |val|=5.000000e+00  ✓ nonzero
    >> break inner loop (พบค่า nonzero แล้ว)
    => row 0: NONZERO ✓  |  rank สะสม = 1

--- row 1: [0.0] (scan 1 cols) ---
    [1,0] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    => row 1: ALL ZERO ✗  |  rank สะสม = 1

--- row 2: [0.0] (scan 1 cols) ---
    [2,0] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    => row 2: ALL ZERO ✗  |  rank สะสม = 1

===== Final rank = 1 =====


## Test Case 4: ค่าใกล้ศูนย์มาก (ทดสอบ tolerance)
ค่า 1e-15 ควร < tolerance → ถือว่าเป็นศูนย์

In [6]:
mat4 = [
    [1e-15, 1e-15],   # ค่าเล็กมาก → ถือว่าศูนย์
    [1e-8,  0.0],     # 1e-8 > 1e-9 → nonzero
    [0.0,   1e-10],   # 1e-10 < 1e-9 → ถือว่าศูนย์
]

# คาดว่า rank = 1 (แค่แถว 1 เท่านั้นที่ nonzero)
result = count_rank(mat4, col_limit=2)
assert result == 1, f"Expected 1, got {result}"

--- row 0: [1e-15, 1e-15] (scan 2 cols) ---
    [0,0] val=1.000000e-15  |val|=1.000000e-15  ✗ zero
    [0,1] val=1.000000e-15  |val|=1.000000e-15  ✗ zero
    => row 0: ALL ZERO ✗  |  rank สะสม = 0

--- row 1: [1e-08, 0.0] (scan 2 cols) ---
    [1,0] val=1.000000e-08  |val|=1.000000e-08  ✓ nonzero
    >> break inner loop (พบค่า nonzero แล้ว)
    => row 1: NONZERO ✓  |  rank สะสม = 1

--- row 2: [0.0, 1e-10] (scan 2 cols) ---
    [2,0] val=0.000000e+00  |val|=0.000000e+00  ✗ zero
    [2,1] val=1.000000e-10  |val|=1.000000e-10  ✗ zero
    => row 2: ALL ZERO ✗  |  rank สะสม = 1

===== Final rank = 1 =====


## Test Case 5: เมทริกซ์ว่าง / แถวว่าง

In [7]:
# เมทริกซ์ว่าง
print("=== empty matrix ===")
result = count_rank([], col_limit=3)
assert result == 0

print()

# แถวว่าง (len=0) → scan 0 คอลัมน์ → ถือว่า zero
print("=== rows with empty lists ===")
mat5 = [[], [1.0], []]
result = count_rank(mat5, col_limit=5)
assert result == 1, f"Expected 1, got {result}"

=== empty matrix ===
===== Final rank = 0 =====

=== rows with empty lists ===
--- row 0: [] (scan 0 cols) ---
    => row 0: ALL ZERO ✗  |  rank สะสม = 0

--- row 1: [1.0] (scan 1 cols) ---
    [1,0] val=1.000000e+00  |val|=1.000000e+00  ✓ nonzero
    >> break inner loop (พบค่า nonzero แล้ว)
    => row 1: NONZERO ✓  |  rank สะสม = 1

--- row 2: [] (scan 0 cols) ---
    => row 2: ALL ZERO ✗  |  rank สะสม = 1

===== Final rank = 1 =====


## สรุป

| Test | สถานการณ์ | col_limit | คำตอบ |
|------|-----------|-----------|-------|
| 1 | Full rank 3×3 | 3 | 3 |
| 2 | มีแถวศูนย์ | 3 | 2 |
| 3 | จำกัด col_limit | 1 | 1 |
| 4 | ค่าใกล้ศูนย์ (tolerance) | 2 | 1 |
| 5 | เมทริกซ์ว่าง / แถวว่าง | - | 0, 1 |